<a href="https://colab.research.google.com/github/Damsiiii/BillFold/blob/main/Cleaning_Finance_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Import the relevant libraries

In [ ]:
import os

# Define paths to your Google Drive folders
# Note: Adjust the base path if these folders are inside specific subdirectories (e.g., 'My Drive/Colab Notebooks/')
drive_base_path = '/content/drive/My Drive/finance_project/data'

folders = {
    'financial_transactions': os.path.join(drive_base_path, 'financial_transactions'),
    'financial_transactions_dataset': os.path.join(drive_base_path, 'financial-transactions-dataset-expenses-and-income'),
    'fiqa_dataset': os.path.join(drive_base_path, 'fiqa_dataset'),
    'alpaca': os.path.join(drive_base_path, 'alpaca')
}

# Verify folder accessibility and list contents
for name, path in folders.items():
    if os.path.exists(path):
        print(f"Successfully accessed {name} at: {path}")
        print(f"Contents: {os.listdir(path)[:5]} (showing up to 5 items)\n")
    else:
        print(f"Could not find {name} at: {path}. Please check if the folder is in 'My Drive' or shared with you.\n")

Successfully accessed financial_transactions at: /content/drive/My Drive/finance_project/data/financial_transactions
Contents: ['synthetic_personal_finance_dataset.csv'] (showing up to 5 items)

Successfully accessed financial_transactions_dataset at: /content/drive/My Drive/finance_project/data/financial-transactions-dataset-expenses-and-income
Contents: ['Expenses_clean.csv', 'Income_clean.csv'] (showing up to 5 items)

Successfully accessed fiqa_dataset at: /content/drive/My Drive/finance_project/data/fiqa_dataset
Contents: ['dataset_dict.json', 'train'] (showing up to 5 items)

Successfully accessed alpaca at: /content/drive/My Drive/finance_project/data/alpaca
Contents: ['.cache', 'train-00000-of-00001-a09b74b3ef9c3b56.parquet', 'data'] (showing up to 5 items)



## Cleaning the dataset

### 1. Financial Transactions Dataset (`synthetic_personal_finance_dataset.csv`)
Let's load the CSV file and check its shape, data types, and null/missing value metrics.

In [ ]:
import pandas as pd
import os

# Load dataset
csv_path1 = os.path.join(
    folders['financial_transactions'],
    'synthetic_personal_finance_dataset.csv'
)
df_fin_trans = pd.read_csv(csv_path1)

total_rows = len(df_fin_trans)

# Inspection table
inspection = pd.DataFrame({
    "Data Type": df_fin_trans.dtypes.astype(str),
    "Null Count": df_fin_trans.isnull().sum(),
    "Missing (%)": (df_fin_trans.isnull().sum() / total_rows * 100).round(2),
    "Duplicate Count": df_fin_trans.apply(lambda x: x.duplicated().sum()),
    "Duplicate (%)": (df_fin_trans.apply(lambda x: x.duplicated().sum()) / total_rows * 100).round(2),
    "Unique Count": df_fin_trans.nunique(dropna=True),
    "Unique (%)": (df_fin_trans.nunique(dropna=True) / total_rows * 100).round(2)
})

# Check mixed/inconsistent Python data types
inconsistent = []

for col in df_fin_trans.columns:
    if df_fin_trans[col].dropna().map(type).nunique() > 1:
        inconsistent.append(col)

print("=== Financial Transactions Dataset Inspection ===")
print("Rows:", total_rows)
print("Columns:", len(df_fin_trans.columns))
print("Duplicate Rows:", df_fin_trans.duplicated().sum())
print("Duplicate Row (%):", round(df_fin_trans.duplicated().mean() * 100, 2))
print("Data Type Inconsistencies:", inconsistent if inconsistent else "None")

display(inspection)

=== Financial Transactions Dataset Inspection ===
Rows: 32424
Columns: 20
Duplicate Rows: 0
Duplicate Row (%): 0.0
Data Type Inconsistencies: None


,Data Type,Null Count,Missing (%),Duplicate Count,Duplicate (%),Unique Count,Unique (%)
user_id,object,0,0.00,0,0.00,32424,100.00
age,int64,0,0.00,32372,99.84,52,0.16
gender,object,0,0.00,32421,99.99,3,0.01
education_level,object,0,0.00,32419,99.98,5,0.02
employment_status,object,0,0.00,32420,99.99,4,0.01
job_title,object,0,0.00,32415,99.97,9,0.03
monthly_income_usd,float64,0,0.00,1975,6.09,30449,93.91
monthly_expenses_usd,float64,0,0.00,1105,3.41,31319,96.59
savings_usd,float64,0,0.00,9,0.03,32415,99.97
has_loan,object,0,0.00,32422,99.99,2,0.01


In [ ]:
# Display first 5 rows
print("=== First 5 Rows of Financial Transactions Dataset ===")
display(df_fin_trans.head(10))

=== First 5 Rows of Financial Transactions Dataset ===


,user_id,age,gender,education_level,employment_status,job_title,monthly_income_usd,monthly_expenses_usd,savings_usd,has_loan,loan_type,loan_amount_usd,loan_term_months,monthly_emi_usd,loan_interest_rate_pct,debt_to_income_ratio,credit_score,savings_to_income_ratio,region,record_date
0,U00001,56,Female,High School,Self-employed,Salesperson,3531.69,1182.59,367655.03,No,NaN,0.00,0,0.00,0.00,0.00,430,8.68,Other,2024-01-09
1,U00002,19,Female,PhD,Employed,Salesperson,3531.73,2367.99,260869.10,Yes,Education,146323.34,36,4953.50,13.33,1.40,543,6.16,North America,2022-02-13
2,U00003,20,Female,Master,Employed,Teacher,2799.49,1003.91,230921.21,No,NaN,0.00,0,0.00,0.00,0.00,754,6.87,Africa,2022-05-12
3,U00004,25,Male,PhD,Employed,Manager,5894.88,4440.12,304815.51,Yes,Business,93242.37,24,4926.57,23.93,0.84,461,4.31,Europe,2023-10-02
4,U00005,53,Female,PhD,Employed,Student,5128.93,4137.61,461509.48,No,NaN,0.00,0,0.00,0.00,0.00,516,7.50,Africa,2021-08-07
5,U00006,62,Male,Master,Employed,Student,4051.00,2244.42,194901.41,Yes,Car,407915.25,12,38219.81,22.21,9.43,718,4.01,North America,2025-02-12
6,U00007,22,Female,Master,Student,Accountant,983.69,332.62,37522.56,Yes,Car,296057.60,180,6773.29,26.95,6.89,761,3.18,Asia,2024-06-01
7,U00008,18,Female,Bachelor,Employed,Accountant,6199.29,4503.15,89107.54,No,NaN,0.00,0,0.00,0.00,0.00,540,1.20,Africa,2024-03-17
8,U00009,49,Female,High School,Self-employed,Doctor,4271.51,3062.12,451923.96,No,NaN,0.00,0,0.00,0.00,0.00,327,8.82,Africa,2023-05-02
9,U00010,61,Male,Master,Employed,Student,5216.18,4369.36,203320.50,Yes,Education,324197.37,360,6782.37,25.09,1.30,558,3.25,Asia,2024-11-23


### 2. Financial Transactions Dataset - Expenses & Income (`Expenses_clean.csv` & `Income_clean.csv`)
Let's load and analyze both CSVs within this dataset.

In [ ]:
import pandas as pd
import os

# ============================================================
# Load Datasets
# ============================================================

expenses_path = os.path.join(
    folders['financial_transactions_dataset'],
    'Expenses_clean.csv'
)

income_path = os.path.join(
    folders['financial_transactions_dataset'],
    'Income_clean.csv'
)

df_expenses = pd.read_csv(expenses_path)
df_income = pd.read_csv(income_path)


# ============================================================
# Function to Inspect Dataset
# ============================================================

def inspect_dataset(df):

    total_rows = len(df)

    # Column-level summary
    summary = pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Null Count": df.isnull().sum(),
        "Missing (%)": (
            df.isnull().sum() / total_rows * 100
        ).round(2),

        "Duplicate Count": df.apply(
            lambda x: x.duplicated().sum()
        ),

        "Duplicate (%)": (
            df.apply(lambda x: x.duplicated().sum())
            / total_rows * 100
        ).round(2),

        "Unique Count": df.nunique(dropna=True),

        "Unique (%)": (
            df.nunique(dropna=True) / total_rows * 100
        ).round(2)
    })

    # Check mixed/inconsistent Python data types
    inconsistent = []

    for col in df.columns:
        non_null = df[col].dropna()

        if non_null.map(type).nunique() > 1:
            inconsistent.append(col)

    # Dataset-level information
    dataset_info = {
        "Rows": total_rows,
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Duplicate Row (%)": round(
            df.duplicated().mean() * 100, 2
        ),
        "Data Type Inconsistencies": (
            inconsistent if inconsistent else "None"
        )
    }

    return dataset_info, summary


# ============================================================
# Inspect Both Datasets
# ============================================================

exp_info, exp_summary = inspect_dataset(df_expenses)
inc_info, inc_summary = inspect_dataset(df_income)


In [ ]:
# Display Results
# ============================================================

print("=== 2a. Expenses Dataset Inspection ===")
print("Rows:", exp_info["Rows"])
print("Columns:", exp_info["Columns"])
print("Duplicate Rows:", exp_info["Duplicate Rows"])
print("Duplicate Row (%):", exp_info["Duplicate Row (%)"])
print(
    "Data Type Inconsistencies:",
    exp_info["Data Type Inconsistencies"]
)

display(exp_summary)


print("\n=== 2b. Income Dataset Inspection ===")
print("Rows:", inc_info["Rows"])
print("Columns:", inc_info["Columns"])
print("Duplicate Rows:", inc_info["Duplicate Rows"])
print("Duplicate Row (%):", inc_info["Duplicate Row (%)"])
print(
    "Data Type Inconsistencies:",
    inc_info["Data Type Inconsistencies"]
)

display(inc_summary)

=== 2a. Expenses Dataset Inspection ===
Rows: 938
Columns: 6
Duplicate Rows: 145
Duplicate Row (%): 15.46
Data Type Inconsistencies: None


,Data Type,Null Count,Missing (%),Duplicate Count,Duplicate (%),Unique Count,Unique (%)
date_time,object,0,0.0,658,70.15,280,29.85
category,object,0,0.0,924,98.51,14,1.49
account,object,0,0.0,935,99.68,3,0.32
amount,float64,0,0.0,860,91.68,78,8.32
currency,object,0,0.0,937,99.89,1,0.11
tags,object,0,0.0,931,99.25,7,0.75



=== 2b. Income Dataset Inspection ===
Rows: 349
Columns: 6
Duplicate Rows: 0
Duplicate Row (%): 0.0
Data Type Inconsistencies: None


,Data Type,Null Count,Missing (%),Duplicate Count,Duplicate (%),Unique Count,Unique (%)
date_time,object,0,0.0,140,40.11,209,59.89
category,object,0,0.0,342,97.99,7,2.01
account,object,0,0.0,344,98.57,5,1.43
amount,float64,0,0.0,219,62.75,130,37.25
currency,object,0,0.0,348,99.71,1,0.29
tags,object,0,0.0,340,97.42,9,2.58


### 3. FiQA Dataset
The FiQA dataset directory contains a `dataset_dict.json` and a `train` folder, indicating a Hugging Face Dataset format. We can load it using the `datasets` library or directly examine the JSON configuration to locate the underlying data split.

In [ ]:
!pip install -q datasets

import pandas as pd
from datasets import load_from_disk

# ============================================================
# Load FiQA Dataset
# ============================================================

fiqa_path = folders['fiqa_dataset']

try:
    fiqa_ds = load_from_disk(fiqa_path)
    df_fiqa = pd.DataFrame(fiqa_ds['train'])

    # ========================================================
    # FiQA Dataset Inspection
    # ========================================================

    total_rows = len(df_fiqa)

    # Column-level data quality summary
    fiqa_summary = pd.DataFrame({
        "Data Type": df_fiqa.dtypes.astype(str),

        "Null Count": df_fiqa.isnull().sum(),

        "Missing (%)": (
            df_fiqa.isnull().sum() / total_rows * 100
        ).round(2),

        "Duplicate Count": df_fiqa.apply(
            lambda x: x.duplicated().sum()
        ),

        "Duplicate (%)": (
            df_fiqa.apply(lambda x: x.duplicated().sum())
            / total_rows * 100
        ).round(2),

        "Unique Count": df_fiqa.nunique(dropna=True),

        "Unique (%)": (
            df_fiqa.nunique(dropna=True) / total_rows * 100
        ).round(2)
    })

    # ========================================================
    # Check Mixed / Inconsistent Python Data Types
    # ========================================================

    fiqa_inconsistent = []

    for col in df_fiqa.columns:
        non_null = df_fiqa[col].dropna()
        if non_null.map(type).nunique() > 1:
            fiqa_inconsistent.append(col)

    # Display Results
    print("=== 3. FiQA Train Split Inspection ===")
    print("Rows:", total_rows)
    print("Columns:", len(df_fiqa.columns))
    print("Duplicate Rows:", df_fiqa.duplicated().sum())
    print("Data Type Inconsistencies:", fiqa_inconsistent if fiqa_inconsistent else "None")
    display(fiqa_summary)

except Exception as e:
    print(f"Could not load or process the dataset: {e}")

=== 3. FiQA Train Split Inspection ===
Rows: 1987
Columns: 2
Duplicate Rows: 0
Data Type Inconsistencies: None


,Data Type,Null Count,Missing (%),Duplicate Count,Duplicate (%),Unique Count,Unique (%)
input,object,0,0.0,0,0.0,1987,100.0
output,object,0,0.0,0,0.0,1987,100.0


### 4. Alpaca Dataset (`train-00000-of-00001-a09b74b3ef9c3b56.parquet`)
Let's load the Alpaca training parquet file and check its metrics.

In [ ]:
# Load Alpaca dataset
alpaca_parquet_path = os.path.join(
    folders['alpaca'],
    'train-00000-of-00001-a09b74b3ef9c3b56.parquet'
)

df_alpaca = pd.read_parquet(alpaca_parquet_path)

# Dataset-level EDA quality metrics
alpaca_quality_metrics = pd.DataFrame({
    'Metric': [
        'Number of Rows',
        'Number of Columns',
        'Duplicate Rows',
        'Total Missing Values',
        'Missing Values (%)',
        'Complete Rows (%)',
        'Unique Rows'
    ],
    'Value': [
        len(df_alpaca),
        len(df_alpaca.columns),
        df_alpaca.duplicated().sum(),
        df_alpaca.isnull().sum().sum(),
        round((df_alpaca.isnull().sum().sum() / df_alpaca.size) * 100, 2),
        round((df_alpaca.notnull().all(axis=1).sum() / len(df_alpaca)) * 100, 2),
        df_alpaca.drop_duplicates().shape[0]
    ]
})

# Column-level quality metrics
alpaca_column_metrics = pd.DataFrame({
    'Column': df_alpaca.columns,
    'Data Type': df_alpaca.dtypes.astype(str).values,
    'Missing Values (Count)': df_alpaca.isnull().sum().values,
    'Missing Values (%)': (
        df_alpaca.isnull().sum() / len(df_alpaca) * 100
    ).round(2).values,
    'Unique Values': df_alpaca.nunique(dropna=True).values
})

print("=== Alpaca Dataset - EDA Quality Metrics ===")
display(alpaca_quality_metrics)

print("=== Alpaca Dataset - Column Quality Metrics ===")
display(alpaca_column_metrics)

=== Alpaca Dataset - EDA Quality Metrics ===


,Metric,Value
0,Number of Rows,52002.0
1,Number of Columns,4.0
2,Duplicate Rows,0.0
3,Total Missing Values,0.0
4,Missing Values (%),0.0
5,Complete Rows (%),100.0
6,Unique Rows,52002.0


=== Alpaca Dataset - Column Quality Metrics ===


,Column,Data Type,Missing Values (Count),Missing Values (%),Unique Values
0,instruction,object,0,0.0,52002
1,input,object,0,0.0,19188
2,output,object,0,0.0,50842
3,text,object,0,0.0,52002


## Cleaning data

# 1st dataset fixes

In [ ]:
# 1. Standardize types
df_fin_trans['record_date'] = pd.to_datetime(df_fin_trans['record_date'])
df_fin_trans['has_loan'] = df_fin_trans['has_loan'].map({'Yes': True, 'No': False}).astype(bool)

# 2. Fill structural loan missingness
df_fin_trans['loan_type'] = df_fin_trans['loan_type'].fillna('None')

# 3. Categorical memory optimization
cat_cols = ['gender', 'education_level', 'employment_status', 'job_title', 'region', 'loan_type']
for col in cat_cols:
    df_fin_trans[col] = df_fin_trans[col].astype('category')

# 4. Correct/verify ratio definitions
df_fin_trans['debt_to_income_ratio'] = (df_fin_trans['monthly_emi_usd'] / df_fin_trans['monthly_income_usd']).round(2)
df_fin_trans['savings_to_monthly_income_ratio'] = (df_fin_trans['savings_usd'] / df_fin_trans['monthly_income_usd']).round(2)

Fix EMI vals

In [ ]:
"""Fix Broken & Inconsistent EMI Values
The Issue: The reported monthly EMI drifts from standard amortization math due to inconsistent rounding and arbitrary formula implementations in the generator.

The Fix: Recompute monthly_emi_usd strictly using the standard fixed-rate amortization formula:"""

import numpy as np

def calculate_emi(row):
    if not row['has_loan'] or row['loan_amount_usd'] <= 0 or row['loan_term_months'] <= 0:
        return 0.0

    p = row['loan_amount_usd']
    n = row['loan_term_months']
    r = (row['loan_interest_rate_pct'] / 100.0) / 12.0

    if r == 0:
        return round(p / n, 2)

    emi = p * (r * (1 + r)**n) / ((1 + r)**n - 1)
    return round(emi, 2)

df_fin_trans['monthly_emi_usd'] = df_fin_trans.apply(calculate_emi, axis=1)

Monthly savings fix

In [ ]:
# Monthly savings-to-income multiplier
df_fin_trans['savings_to_monthly_income_ratio'] = np.where(
    df_fin_trans['monthly_income_usd'] > 0,
    (df_fin_trans['savings_usd'] / df_fin_trans['monthly_income_usd']).round(2),
    np.nan
)

# Annual savings-to-income multiplier (matching the original column's magnitude)
df_fin_trans['savings_to_annual_income_ratio'] = np.where(
    df_fin_trans['monthly_income_usd'] > 0,
    (df_fin_trans['savings_usd'] / (df_fin_trans['monthly_income_usd'] * 12)).round(2),
    np.nan
)

# Drop or replace the ambiguous legacy column
df_fin_trans.drop(columns=['savings_to_income_ratio'], inplace=True)

In [ ]:
# Recompute debt_to_income_ratio (DTI)
df_fin_trans['debt_to_income_ratio'] = np.where(
    df_fin_trans['monthly_income_usd'] > 0,
    (df_fin_trans['monthly_emi_usd'] / df_fin_trans['monthly_income_usd']).round(4),
    0.0
)

In [ ]:
## Fix Cash Flow Decoupling (Expenses vs. EMI)

# 1. Clarify that expenses represent living/non-debt expenses
df_fin_trans.rename(columns={'monthly_expenses_usd': 'monthly_living_expenses_usd'}, inplace=True)

# 2. Total monthly financial commitments
df_fin_trans['total_monthly_obligations_usd'] = (
    df_fin_trans['monthly_living_expenses_usd'] + df_fin_trans['monthly_emi_usd']
).round(2)

# 3. True disposable cash flow surplus/deficit
df_fin_trans['net_monthly_cash_flow_usd'] = (
    df_fin_trans['monthly_income_usd'] - df_fin_trans['total_monthly_obligations_usd']
).round(2)

In [ ]:
# Convert to object first so pandas doesn't coerce None back into NaN
df_fin_trans['loan_interest_rate_pct'] = df_fin_trans['loan_interest_rate_pct'].astype(object)
df_fin_trans['loan_term_months'] = df_fin_trans['loan_term_months'].astype(object)

non_borrower_mask = ~df_fin_trans['has_loan']

df_fin_trans.loc[non_borrower_mask, 'loan_interest_rate_pct'] = 'None'
df_fin_trans.loc[non_borrower_mask, 'loan_term_months'] = 'None'

In [ ]:
# record_date is stored as an unparsed object string with trailing timestamps, and low-cardinality fields waste memory.
# The Fix: Cast timestamps to datetime64[ns] and categorical variables to category:
df_fin_trans['record_date'] = pd.to_datetime(df_fin_trans['record_date'])

categorical_cols = ['gender', 'education_level', 'employment_status', 'job_title', 'region', 'loan_type']
for col in categorical_cols:
    df_fin_trans[col] = df_fin_trans[col].astype('category')

In [ ]:
display(df_fin_trans.head(10))

,user_id,age,gender,education_level,employment_status,job_title,monthly_income_usd,monthly_living_expenses_usd,savings_usd,has_loan,...,monthly_emi_usd,loan_interest_rate_pct,debt_to_income_ratio,credit_score,region,record_date,savings_to_monthly_income_ratio,savings_to_annual_income_ratio,total_monthly_obligations_usd,net_monthly_cash_flow_usd
0,U00001,56,Female,High School,Self-employed,Salesperson,3531.69,1182.59,367655.03,False,...,0.00,None,0.0000,430,Other,2024-01-09,104.10,8.68,1182.59,2349.10
1,U00002,19,Female,PhD,Employed,Salesperson,3531.73,2367.99,260869.10,True,...,4953.50,13.33,1.4026,543,North America,2022-02-13,73.86,6.16,7321.49,-3789.76
2,U00003,20,Female,Master,Employed,Teacher,2799.49,1003.91,230921.21,False,...,0.00,None,0.0000,754,Africa,2022-05-12,82.49,6.87,1003.91,1795.58
3,U00004,25,Male,PhD,Employed,Manager,5894.88,4440.12,304815.51,True,...,4926.57,23.93,0.8357,461,Europe,2023-10-02,51.71,4.31,9366.69,-3471.81
4,U00005,53,Female,PhD,Employed,Student,5128.93,4137.61,461509.48,False,...,0.00,None,0.0000,516,Africa,2021-08-07,89.98,7.50,4137.61,991.32
5,U00006,62,Male,Master,Employed,Student,4051.00,2244.42,194901.41,True,...,38219.81,22.21,9.4347,718,North America,2025-02-12,48.11,4.01,40464.23,-36413.23
6,U00007,22,Female,Master,Student,Accountant,983.69,332.62,37522.56,True,...,6773.29,26.95,6.8856,761,Asia,2024-06-01,38.14,3.18,7105.91,-6122.22
7,U00008,18,Female,Bachelor,Employed,Accountant,6199.29,4503.15,89107.54,False,...,0.00,None,0.0000,540,Africa,2024-03-17,14.37,1.20,4503.15,1696.14
8,U00009,49,Female,High School,Self-employed,Doctor,4271.51,3062.12,451923.96,False,...,0.00,None,0.0000,327,Africa,2023-05-02,105.80,8.82,3062.12,1209.39
9,U00010,61,Male,Master,Employed,Student,5216.18,4369.36,203320.50,True,...,6782.37,25.09,1.3003,558,Asia,2024-11-23,38.98,3.25,11151.73,-5935.55


In [ ]:
import os

# Define the target directory path
target_dir = '/content/drive/MyDrive/finance_project/data/Damsara/1st dataset/'

# Create the directory if it does not exist
os.makedirs(target_dir, exist_ok=True)

# Define the final file path
save_path = os.path.join(target_dir, 'cleaned_synthetic_personal_finance_dataset.csv')

# Save the cleaned DataFrame
df_fin_trans.to_csv(save_path, index=False)
print(f"Cleaned dataset successfully saved to: {save_path}")

Cleaned dataset successfully saved to: /content/drive/MyDrive/finance_project/data/Damsara/1st dataset/cleaned_synthetic_personal_finance_dataset.csv


## Second dataset

In [ ]:
import os
import pandas as pd

# Define paths for the second dataset
expenses_path = os.path.join(folders['financial_transactions_dataset'], 'Expenses_clean.csv')
income_path = os.path.join(folders['financial_transactions_dataset'], 'Income_clean.csv')

# Load datasets
df_expenses_clean = pd.read_csv(expenses_path)
df_income_clean = pd.read_csv(income_path)

In [ ]:
display(df_expenses_clean.head(10))

,date_time,category,account,amount,currency,tags
0,2025-11-30 00:00:00,Health,acct_1,114.0,BYN,tag_1
1,2025-11-29 00:00:00,Food,acct_1,5.0,BYN,tag_1
2,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
3,2025-11-27 00:00:00,Cafe,acct_1,10.0,BYN,tag_1
4,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
5,2025-11-26 00:00:00,Cafe,acct_3,2.0,BYN,tag_1
6,2025-11-26 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
7,2025-11-25 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
8,2025-11-25 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
9,2025-11-25 00:00:00,Cafe,acct_1,11.0,BYN,tag_1


In [ ]:
display(df_income_clean.head(10))

,date_time,category,account,amount,currency,tags
0,2025-11-29 00:00:00,Job,acct_1,49.0,BYN,tag_1
1,2025-11-29 00:00:00,Job,acct_1,18.0,BYN,tag_2
2,2025-11-29 00:00:00,Job,acct_1,32.0,BYN,tag_3
3,2025-11-29 00:00:00,Job,acct_1,109.0,BYN,tag_4
4,2025-11-28 00:00:00,Second work,acct_1,132.0,BYN,2th_work
5,2025-11-25 00:00:00,Debt return / Borrowed money,acct_1,49.0,BYN,tag_5
6,2025-11-25 00:00:00,Job,acct_1,18.0,BYN,tag_1
7,2025-11-22 00:00:00,Job,acct_1,44.0,BYN,tag_1
8,2025-11-22 00:00:00,Job,acct_1,10.0,BYN,tag_2
9,2025-11-22 00:00:00,Job,acct_1,11.0,BYN,tag_3


In [ ]:
# 1. Clean Expenses Dataset
# Parse date_time as datetime
df_expenses_clean['date_time'] = pd.to_datetime(df_expenses_clean['date_time'])
# Drop duplicate rows (145 duplicates were detected during inspection)
df_expenses_clean = df_expenses_clean.drop_duplicates().reset_index(drop=True)

In [ ]:
# 2. Clean Income Dataset
# Parse date_time as datetime
df_income_clean['date_time'] = pd.to_datetime(df_income_clean['date_time'])

In [ ]:
# 3. Create target directory and save cleaned files
target_dir_2nd = '/content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/'
os.makedirs(target_dir_2nd, exist_ok=True)

save_path_expenses = os.path.join(target_dir_2nd, 'cleaned_expenses.csv')
save_path_income = os.path.join(target_dir_2nd, 'cleaned_income.csv')

df_expenses_clean.to_csv(save_path_expenses, index=False)
df_income_clean.to_csv(save_path_income, index=False)

print(f"Cleaned expenses saved to: {save_path_expenses} (Shape: {df_expenses_clean.shape})")
print(f"Cleaned income saved to: {save_path_income} (Shape: {df_income_clean.shape})")

Cleaned expenses saved to: /content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/cleaned_expenses.csv (Shape: (793, 6))
Cleaned income saved to: /content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/cleaned_income.csv (Shape: (349, 6))


In [ ]:
# Verify and ensure all cleaned datasets are saved properly in their respective target directories
import os

target_dir_1st = '/content/drive/MyDrive/finance_project/data/Damsara/1st dataset/'
target_dir_2nd = '/content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/'

# Ensure directories exist
os.makedirs(target_dir_1st, exist_ok=True)
os.makedirs(target_dir_2nd, exist_ok=True)

# Define file paths
path_1st = os.path.join(target_dir_1st, 'cleaned_synthetic_personal_finance_dataset.csv')
path_expenses = os.path.join(target_dir_2nd, 'cleaned_expenses.csv')
path_income = os.path.join(target_dir_2nd, 'cleaned_income.csv')

# Save the datasets
df_fin_trans.to_csv(path_1st, index=False)
df_expenses_clean.to_csv(path_expenses, index=False)
df_income_clean.to_csv(path_income, index=False)

print("=== Saving Verification ===")
print(f"1st Dataset Saved to: {path_1st} | Shape: {df_fin_trans.shape}")
print(f"2nd Dataset (Expenses) Saved to: {path_expenses} | Shape: {df_expenses_clean.shape}")
print(f"2nd Dataset (Income) Saved to: {path_income} | Shape: {df_income_clean.shape}")


=== Saving Verification ===
1st Dataset Saved to: /content/drive/MyDrive/finance_project/data/Damsara/1st dataset/cleaned_synthetic_personal_finance_dataset.csv | Shape: (32424, 23)
2nd Dataset (Expenses) Saved to: /content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/cleaned_expenses.csv | Shape: (793, 6)
2nd Dataset (Income) Saved to: /content/drive/MyDrive/finance_project/data/Damsara/2nd dataset/cleaned_income.csv | Shape: (349, 6)
